# 1.4 Dense Layer 與 Activation

這份 Notebook 先畫出常見 activation function，再用 XOR 任務比較線性模型與 ReLU DNN 的差異。


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)


## 1. 觀察常見 Activation Function


In [ ]:
x = np.linspace(-5, 5, 300).astype('float32')
activations = {
    'linear': x,
    'sigmoid': tf.keras.activations.sigmoid(x).numpy(),
    'tanh': tf.keras.activations.tanh(x).numpy(),
    'relu': tf.keras.activations.relu(x).numpy()
}

plt.figure(figsize=(7, 4))
for name, values in activations.items():
    plt.plot(x, values, label=name)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.legend()
plt.title('Activation Functions')
plt.show()


## 2. 建立 XOR 非線性分類資料


In [ ]:
rng = np.random.default_rng(42)
X = rng.uniform(-1, 1, size=(1200, 2)).astype('float32')
y = ((X[:, 0] * X[:, 1]) > 0).astype('float32')
X += rng.normal(0, 0.08, size=X.shape).astype('float32')

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

plt.figure(figsize=(5, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap='coolwarm', s=12)
plt.title('XOR-like dataset')
plt.show()


## 3. 線性模型：沒有隱藏層


In [ ]:
linear_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
linear_model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='binary_crossentropy', metrics=['accuracy'])
linear_history = linear_model.fit(x_train, y_train, validation_split=0.2, epochs=80, batch_size=32, verbose=0)

linear_train = linear_model.evaluate(x_train, y_train, verbose=0, return_dict=True)
linear_test = linear_model.evaluate(x_test, y_test, verbose=0, return_dict=True)
pd.DataFrame([linear_train, linear_test], index=['linear_train', 'linear_test'])


## 4. ReLU DNN：加入非線性隱藏層


In [ ]:
relu_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
relu_model.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='binary_crossentropy', metrics=['accuracy'])
relu_history = relu_model.fit(x_train, y_train, validation_split=0.2, epochs=80, batch_size=32, verbose=0)

relu_train = relu_model.evaluate(x_train, y_train, verbose=0, return_dict=True)
relu_test = relu_model.evaluate(x_test, y_test, verbose=0, return_dict=True)
pd.DataFrame([relu_train, relu_test], index=['relu_train', 'relu_test'])


## 5. 比較結果


In [ ]:
comparison = pd.DataFrame([
    {'model': 'Linear sigmoid', 'train_acc': linear_train['accuracy'], 'test_acc': linear_test['accuracy']},
    {'model': 'ReLU DNN', 'train_acc': relu_train['accuracy'], 'test_acc': relu_test['accuracy']},
])
comparison


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(linear_history.history['val_accuracy'], label='linear val acc')
plt.plot(relu_history.history['val_accuracy'], label='relu dnn val acc')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.title('Activation Makes Nonlinear Learning Possible')
plt.show()


## 6. 小結

Dense layer 做加權組合，activation function 提供非線性能力。對 XOR 這類非線性任務，沒有隱藏層的線性模型表現有限，而 ReLU DNN 可以明顯改善。
